# Train neural-only IPN 13 nhãn cải tiến trên Google Colab

Notebook này dùng bộ keypoint IPN 13 nhãn trên Google Drive, chọn checkpoint bằng validation macro-F1 và giữ test split ngoài quá trình tuning.

Trước khi chạy: chọn **Runtime → Change runtime type → T4 GPU**. Mặc định notebook đọc `MyDrive/ipn-extract-data`, gồm `train.txt`, `val.txt`, `test.txt` và cache 13 nhãn tại `keypoint_tensor_cache/T64/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, time
from collections import deque
from pathlib import Path
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU: Runtime > Change runtime type > T4 GPU')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Cấu hình

Notebook tự clone code từ GitHub. Google Drive chỉ cần chứa keypoint cache và các file split; không cần upload source code hay project lên Drive.

In [ ]:
# ===== CHỈNH CÁC GIÁ TRỊ NÀY =====
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
CACHE_ROOT_ON_DRIVE = DRIVE_DATA_ROOT / 'keypoint_tensor_cache'
TRAIN_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'val.txt'
TEST_FILE_ON_DRIVE = DRIVE_DATA_ROOT / 'test.txt'
OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'neural_improved_13cls'

REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
GIT_BRANCH = 'HuyenHoang'

COPY_CACHE_TO_COLAB = False  # đọc trực tiếp từ Drive; bật True nếu muốn tăng tốc I/O
NUM_CLASSES = 13
NUM_FRAMES = 64
EPOCHS = 50
BATCH_SIZE = 64  # cấu hình an toàn cho T4; tăng lên 128 nếu còn nhiều VRAM
NUM_WORKERS = 0  # đọc trực tiếp Google Drive ổn định hơn multiprocessing
D_MODEL = 128
LAYERS = 3
HEADS = 4
BACKBONE = 'transformer'  # transformer, tcn, bigru, mlp_mixer
SEEDS = [42, 123, 3407]
RUN_ALL_SEEDS = False  # chạy seed đầu trước; bật True sau khi cấu hình đã ổn

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Chuẩn bị code

In [ ]:
PROJECT_ROOT = Path('/content/dynamic_hand_gesture_recognition')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', GIT_BRANCH], check=True)

NEURAL_DIR = PROJECT_ROOT / 'final' / 'code' / 'neural_only_experiments'
TRAIN_SCRIPT = NEURAL_DIR / 'train_neural_only.py'
TEST_SCRIPT = NEURAL_DIR / 'test_neural_only.py'
for required in (TRAIN_SCRIPT, TEST_SCRIPT):
    if not required.exists():
        raise FileNotFoundError(required)

train_source = TRAIN_SCRIPT.read_text(encoding='utf-8')
if '--select-metric' not in train_source or '--disable-motion-features' not in train_source:
    raise RuntimeError('Branch GitHub chưa có code neural cải tiến.')
print('Project:', PROJECT_ROOT)
print('Commit:', subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

## 3. Kiểm tra và copy dữ liệu

Copy cache vào ổ tạm của Colab giúp DataLoader nhanh hơn. Output/checkpoint vẫn được ghi trực tiếp về Drive.

In [ ]:
for path in (CACHE_ROOT_ON_DRIVE, TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE):
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy: {path}')

if COPY_CACHE_TO_COLAB:
    LOCAL_DATA_ROOT = Path('/content/neural_data')
    LOCAL_CACHE_ROOT = LOCAL_DATA_ROOT / 'keypoint_tensor_cache'
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not LOCAL_CACHE_ROOT.exists():
        print('Đang copy cache từ Drive sang Colab...')
        shutil.copytree(CACHE_ROOT_ON_DRIVE, LOCAL_CACHE_ROOT)
    for source in (TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE, TEST_FILE_ON_DRIVE):
        if source.exists():
            shutil.copy2(source, LOCAL_DATA_ROOT / source.name)
    CACHE_ROOT = LOCAL_CACHE_ROOT
    TRAIN_FILE = LOCAL_DATA_ROOT / TRAIN_FILE_ON_DRIVE.name
    VAL_FILE = LOCAL_DATA_ROOT / VAL_FILE_ON_DRIVE.name
    TEST_FILE = LOCAL_DATA_ROOT / TEST_FILE_ON_DRIVE.name
else:
    CACHE_ROOT = CACHE_ROOT_ON_DRIVE
    TRAIN_FILE, VAL_FILE, TEST_FILE = TRAIN_FILE_ON_DRIVE, VAL_FILE_ON_DRIVE, TEST_FILE_ON_DRIVE

print('Cache:', CACHE_ROOT)
print('Train:', TRAIN_FILE)
print('Val:', VAL_FILE)
print('Test:', TEST_FILE)

In [ ]:
sys.path.insert(0, str(NEURAL_DIR))
from dataset import NeuralOnlyKeypointDataset
from train_neural_only import class_weights, validate_no_split_overlap

train_set = NeuralOnlyKeypointDataset(TRAIN_FILE, CACHE_ROOT, 'train', NUM_FRAMES)
val_set = NeuralOnlyKeypointDataset(VAL_FILE, CACHE_ROOT, 'val', NUM_FRAMES)
validate_no_split_overlap(train_set.samples, val_set.samples)
if NUM_CLASSES != 13:
    raise ValueError(f'NUM_CLASSES={NUM_CLASSES} đang bị giữ từ phiên cũ. Hãy Restart session rồi Run all để nạp NUM_CLASSES=13.')
train_label_ids = sorted(set(label for _, label in train_set.samples))
val_label_ids = sorted(set(label for _, label in val_set.samples))
expected_label_ids = list(range(NUM_CLASSES))
if train_label_ids != expected_label_ids:
    raise ValueError(f'Train label IDs không khớp: actual={train_label_ids}, expected={expected_label_ids}')
if not set(val_label_ids).issubset(expected_label_ids):
    raise ValueError(f'Validation có label ngoài 0..12: {val_label_ids}')
weights = class_weights(train_set.samples, NUM_CLASSES, 0.5)

print('Train samples:', len(train_set), 'Val samples:', len(val_set))
print('Train classes:', len(train_label_ids), 'IDs:', train_label_ids)
print('Val classes:', len(val_label_ids), 'IDs:', val_label_ids)
print('Tensor shape:', tuple(train_set[0]['keypoints'].shape))
print('Class weight range:', float(weights.min()), '→', float(weights.max()))

## 4. Train

Lần đầu nên để `RUN_ALL_SEEDS=False`. Khi một seed chạy ổn, bật lại để có mean ± std đáng tin cậy.

In [ ]:
def run_train(seed: int) -> Path:
    output_dir = OUTPUT_ROOT / f'{BACKBONE}_seed{seed}'
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-u', str(TRAIN_SCRIPT),
        '--backbone', BACKBONE,
        '--cache-root', str(CACHE_ROOT),
        '--train-file', str(TRAIN_FILE),
        '--val-file', str(VAL_FILE),
        '--output-dir', str(output_dir),
        '--num-classes', str(NUM_CLASSES),
        '--num-frames', str(NUM_FRAMES),
        '--batch-size', str(BATCH_SIZE),
        '--epochs', str(EPOCHS),
        '--d-model', str(D_MODEL),
        '--layers', str(LAYERS),
        '--heads', str(HEADS),
        '--dropout', '0.15',
        '--label-smoothing', '0.05',
        '--class-weight-power', '0.5',
        '--jitter-std', '0.005',
        '--frame-mask-probability', '0.05',
        '--keypoint-dropout-probability', '0.03',
        '--select-metric', 'macro_f1',
        '--early-stop-patience', '8',
        '--seed', str(seed),
        '--device', 'cuda:0',
        '--num-workers', str(NUM_WORKERS),
        '--log-every', '20',
    ]
    (output_dir / 'command.json').write_text(json.dumps(command, indent=2), encoding='utf-8')
    print('\nRUN:', ' '.join(command))
    log_path = output_dir / 'train.log'
    tail = deque(maxlen=50)
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
            tail.append(line.rstrip())
        return_code = process.wait()
    if return_code != 0:
        error_tail = '\n'.join(tail)
        raise RuntimeError(f'Train thất bại (exit={return_code}). Log: {log_path}\n{error_tail}')
    return output_dir

seeds_to_run = SEEDS if RUN_ALL_SEEDS else SEEDS[:1]
trained_dirs = [run_train(seed) for seed in seeds_to_run]
print('Hoàn tất:', trained_dirs)

## 5. Vẽ learning curve

In [ ]:
import matplotlib.pyplot as plt

for output_dir in trained_dirs:
    history = json.loads((output_dir / 'history.json').read_text(encoding='utf-8'))
    epochs = [row['epoch'] for row in history]
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, [row['train']['accuracy'] for row in history], label='train acc')
    plt.plot(epochs, [row['val']['accuracy'] for row in history], label='val acc')
    plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, [row['val']['macro_f1'] for row in history], label='val macro-F1')
    plt.xlabel('epoch'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(output_dir.name)
    plt.tight_layout(); plt.show()

## 6. Đánh giá test đúng một lần

Chỉ chạy cell này sau khi đã chốt cấu hình và seed bằng validation. Không chỉnh hyperparameter dựa trên test.

Mỗi seed được đánh giá độc lập từ `OUTPUT_ROOT / f'{BACKBONE}_seed{seed}'`, không phụ thuộc biến `trained_dirs` còn trong RAM. Kết quả nằm trong thư mục `locked_test/` riêng của seed: `test_metrics.json`, `predictions.csv`, `test_per_class_metrics.csv`, `test_confusion_matrix.csv`, `test_report.md`, `provenance.json` và `LOCKED_TEST_COMPLETE.json`. Per-class accuracy trong các artifact là recall (đúng / tổng số mẫu thật của lớp).

Mặc định cell từ chối chạy đè nếu thư mục locked test đã có artifact hoặc marker. Chỉ đặt `ALLOW_LOCKED_TEST_OVERWRITE=True` khi chủ động tạo lại toàn bộ thư mục locked test.

In [ ]:
RUN_LOCKED_TEST = False
FINAL_TEST_SEEDS = SEEDS if RUN_ALL_SEEDS else SEEDS[:1]
ALLOW_LOCKED_TEST_OVERWRITE = False

if RUN_LOCKED_TEST:
    if not TEST_FILE.exists():
        raise FileNotFoundError(TEST_FILE)
    import csv
    import hashlib
    from datetime import datetime, timezone

    def _class_id(row, fields, label_names):
        for field in fields:
            value = row.get(field)
            if value not in (None, ''):
                try:
                    numeric_value = float(value)
                except (TypeError, ValueError):
                    continue
                if not numeric_value.is_integer():
                    raise ValueError(f'Class ID in {field} is not integral: {value}')
                return int(numeric_value)
        for field in ('true_label', 'pred_label', 'label', 'class_name', 'name'):
            value = row.get(field)
            if value not in (None, ''):
                for class_id, label_name in label_names.items():
                    if str(label_name) == str(value):
                        return class_id
        return None

    def _row_class_id(row):
        for field in ('class_id', 'label_id', 'id', 'index'):
            value = row.get(field)
            if value not in (None, ''):
                try:
                    numeric_value = float(value)
                except (TypeError, ValueError):
                    continue
                if not numeric_value.is_integer():
                    raise ValueError(f'Class ID in {field} is not integral: {value}')
                return int(numeric_value)
        return None

    def _sha256(path):
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()

    git_commit = subprocess.check_output(
        ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True
    ).strip()
    test_file_sha256 = _sha256(TEST_FILE)

    for seed in FINAL_TEST_SEEDS:
        output_dir = OUTPUT_ROOT / f'{BACKBONE}_seed{seed}'
        checkpoint_path = output_dir / 'best.pt'
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f'Không tìm thấy checkpoint đã chốt: {checkpoint_path}')
        locked_dir = output_dir / 'locked_test'
        metrics_path = locked_dir / 'test_metrics.json'
        predictions_path = locked_dir / 'predictions.csv'
        legacy_predictions_path = locked_dir / 'test_predictions.csv'
        per_class_path = locked_dir / 'test_per_class_metrics.csv'
        confusion_path = locked_dir / 'test_confusion_matrix.csv'
        report_path = locked_dir / 'test_report.md'
        provenance_path = locked_dir / 'provenance.json'
        completion_marker = locked_dir / 'LOCKED_TEST_COMPLETE.json'
        expected_paths = [
            metrics_path, predictions_path, legacy_predictions_path, per_class_path,
            confusion_path, report_path, provenance_path, completion_marker,
        ]
        existing_paths = [path for path in expected_paths if path.exists()]
        if locked_dir.exists() and locked_dir.is_dir():
            existing_paths.extend(path for path in locked_dir.iterdir() if path not in existing_paths)
        elif locked_dir.exists():
            existing_paths.append(locked_dir)
        if existing_paths and not ALLOW_LOCKED_TEST_OVERWRITE:
            raise FileExistsError(
                'Locked test output already exists; set ALLOW_LOCKED_TEST_OVERWRITE=True '
                f'to recreate {locked_dir}: {existing_paths}'
            )
        if ALLOW_LOCKED_TEST_OVERWRITE and locked_dir.exists():
            if locked_dir.is_dir():
                shutil.rmtree(locked_dir)
            else:
                locked_dir.unlink()
        locked_dir.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable, '-u', str(TEST_SCRIPT),
            '--checkpoint', str(checkpoint_path),
            '--cache-root', str(CACHE_ROOT),
            '--test-file', str(TEST_FILE),
            '--batch-size', str(BATCH_SIZE),
            '--num-workers', str(NUM_WORKERS),
            '--device', 'cuda:0',
            '--output', str(metrics_path),
        ]
        subprocess.run(command, check=True)

        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        label_names = {}
        raw_label_names = metrics.get('label_names', {})
        if isinstance(raw_label_names, dict):
            for class_id, label_name in raw_label_names.items():
                try:
                    numeric_class_id = float(class_id)
                except (TypeError, ValueError):
                    continue
                if not numeric_class_id.is_integer():
                    raise ValueError(f'Class ID in label_names is not integral: {class_id}')
                label_names[int(numeric_class_id)] = str(label_name)
        metric_rows = metrics.get('per_class', [])
        if isinstance(metric_rows, dict):
            metric_rows = list(metric_rows.values())
        if isinstance(metric_rows, list):
            for metric_row in metric_rows:
                if not isinstance(metric_row, dict):
                    continue
                class_id = _row_class_id(metric_row)
                if class_id is None:
                    continue
                label_name = metric_row.get('label_name', metric_row.get('class_name'))
                if label_name not in (None, ''):
                    label_names[class_id] = str(label_name)

        if not predictions_path.exists() and legacy_predictions_path.exists():
            predictions_path = legacy_predictions_path
        if not predictions_path.exists():
            raise FileNotFoundError(f'Không tìm thấy predictions.csv trong {locked_dir}')
        with predictions_path.open('r', encoding='utf-8-sig', newline='') as handle:
            prediction_rows = list(csv.DictReader(handle))
        if not prediction_rows:
            raise ValueError(f'predictions.csv rỗng: {predictions_path}')

        parsed_predictions = []
        for prediction_row in prediction_rows:
            true_id = _class_id(
                prediction_row,
                ('true_id', 'target_id', 'label_id', 'true', 'target', 'label', 'y_true'),
                label_names,
            )
            pred_id = _class_id(
                prediction_row,
                ('pred_id', 'prediction_id', 'predicted_id', 'pred', 'prediction', 'y_pred'),
                label_names,
            )
            if true_id is None or pred_id is None:
                raise ValueError(f'Không đọc được true/pred class ID từ row: {prediction_row}')
            for class_id, field in ((true_id, 'true_label'), (pred_id, 'pred_label')):
                label_name = prediction_row.get(field)
                if label_name not in (None, ''):
                    try:
                        int(float(label_name))
                    except (TypeError, ValueError):
                        label_names.setdefault(class_id, str(label_name))
            parsed_predictions.append((true_id, pred_id))

        class_counts = [NUM_CLASSES]
        for field in ('num_classes', 'n_classes'):
            try:
                class_counts.append(int(metrics[field]))
            except (KeyError, TypeError, ValueError):
                pass
        if isinstance(metric_rows, list):
            class_counts.append(len(metric_rows))
        old_confusion = metrics.get('confusion_matrix', [])
        if isinstance(old_confusion, list):
            class_counts.append(len(old_confusion))
        max_seen_id = max(item for pair in parsed_predictions for item in pair)
        num_classes = max(max(class_counts), max_seen_id + 1)
        confusion = [[0 for _ in range(num_classes)] for _ in range(num_classes)]
        for true_id, pred_id in parsed_predictions:
            if not (0 <= true_id < num_classes and 0 <= pred_id < num_classes):
                raise ValueError(f'Class ID ngoài phạm vi 0..{num_classes - 1}: {true_id}, {pred_id}')
            confusion[true_id][pred_id] += 1

        per_class_rows = []
        for class_id in range(num_classes):
            support = sum(row_true == class_id for row_true, _ in parsed_predictions)
            predicted = sum(row_pred == class_id for _, row_pred in parsed_predictions)
            correct = confusion[class_id][class_id]
            recall = correct / support if support else 0.0
            precision = correct / predicted if predicted else 0.0
            f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
            per_class_rows.append({
                'class_id': class_id,
                'label_name': label_names.get(class_id, str(class_id)),
                'support': support,
                'correct': correct,
                'predicted': predicted,
                'accuracy': recall,
                'recall': recall,
                'precision': precision,
                'f1': f1,
            })

        total = len(parsed_predictions)
        correct_total = sum(row['correct'] for row in per_class_rows)
        accuracy = correct_total / total if total else 0.0
        macro_precision = sum(row['precision'] for row in per_class_rows) / max(num_classes, 1)
        macro_recall = sum(row['recall'] for row in per_class_rows) / max(num_classes, 1)
        macro_f1 = sum(row['f1'] for row in per_class_rows) / max(num_classes, 1)
        present_rows = [row for row in per_class_rows if row['support'] > 0]
        balanced_accuracy = sum(row['recall'] for row in present_rows) / max(len(present_rows), 1)

        per_class_fields = [
            'class_id', 'label_name', 'support', 'correct', 'predicted',
            'accuracy', 'recall', 'precision', 'f1',
        ]
        with per_class_path.open('w', encoding='utf-8', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=per_class_fields)
            writer.writeheader()
            writer.writerows(per_class_rows)

        with confusion_path.open('w', encoding='utf-8', newline='') as handle:
            writer = csv.writer(handle)
            writer.writerow(['true_id', 'true_label'] + [f'pred_{class_id}' for class_id in range(num_classes)])
            for class_id, matrix_row in enumerate(confusion):
                writer.writerow([class_id, label_names.get(class_id, str(class_id))] + matrix_row)

        report_lines = [
            '# Locked test report',
            '',
            f'- Locked test directory: `{locked_dir}`',
            f'- Accuracy: `{accuracy:.4f}` ({correct_total}/{total})',
            f'- Macro precision: `{macro_precision:.4f}`',
            f'- Macro recall: `{macro_recall:.4f}`',
            f'- Macro-F1: `{macro_f1:.4f}`',
            f'- Balanced accuracy: `{balanced_accuracy:.4f}`',
            '',
            '## Artifacts',
            f'- Metrics JSON: `{metrics_path}`',
            f'- Predictions CSV: `{predictions_path}`',
            f'- Per-class metrics CSV: `{per_class_path}`',
            f'- Confusion matrix CSV (true rows × predicted columns): `{confusion_path}`',
            f'- This report: `{report_path}`',
            f'- Provenance JSON: `{provenance_path}`',
            f'- Completion marker: `{completion_marker}`',
            '',
            '## Per-class metrics',
            '| ID | Label | Support | Correct | Predicted | Accuracy/Recall | Precision | F1 |',
            '|---:|:---|---:|---:|---:|---:|---:|---:|',
        ]
        for row in per_class_rows:
            report_lines.append(
                '| {} | {} | {} | {} | {} | {:.4f} | {:.4f} | {:.4f} |'.format(
                    row['class_id'], row['label_name'], row['support'], row['correct'],
                    row['predicted'], row['accuracy'], row['precision'], row['f1'],
                )
            )
        report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

        artifact_paths = {
            'metrics': str(metrics_path),
            'predictions': str(predictions_path),
            'per_class_metrics': str(per_class_path),
            'confusion_matrix': str(confusion_path),
            'report': str(report_path),
            'provenance': str(provenance_path),
            'completion_marker': str(completion_marker),
        }
        provenance = {
            'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            'seed': seed,
            'backbone': BACKBONE,
            'git_commit': git_commit,
            'checkpoint': str(checkpoint_path),
            'checkpoint_sha256': _sha256(checkpoint_path),
            'test_file': str(TEST_FILE),
            'test_file_sha256': test_file_sha256,
            'command': command,
            'artifact_paths': artifact_paths,
        }
        provenance_path.write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding='utf-8')
        completion_marker.write_text(
            json.dumps({'complete': True, **provenance}, indent=2, ensure_ascii=False),
            encoding='utf-8',
        )

        print(f'\nPer-class results for {locked_dir}:')
        print('  ID | Label | Support | Correct | Predicted | Accuracy/Recall | Precision | F1')
        for row in per_class_rows:
            print('  {} | {} | {} | {} | {} | {:.4f} | {:.4f} | {:.4f}'.format(
                row['class_id'], row['label_name'], row['support'], row['correct'],
                row['predicted'], row['accuracy'], row['precision'], row['f1'],
            ))
        print(json.dumps({
            'output_dir': str(output_dir),
            'locked_test_dir': str(locked_dir),
            'metrics': str(metrics_path),
            'predictions': str(predictions_path),
            'per_class_metrics': str(per_class_path),
            'confusion_matrix': str(confusion_path),
            'report': str(report_path),
            'provenance': str(provenance_path),
            'completion_marker': str(completion_marker),
        }, indent=2, ensure_ascii=False))
else:
    print('Test đang khóa. Đặt RUN_LOCKED_TEST=True sau khi đã chốt cấu hình bằng validation.')

## Gợi ý xử lý lỗi

- CUDA out of memory: giảm `BATCH_SIZE` từ 64 xuống 32 hoặc 16.
- Train lỗi: xem `train.log` trong output của seed; notebook luôn hiện 50 dòng lỗi cuối.
- Không tìm thấy cache: kiểm tra có thư mục `T64` và cấu trúc class/split bên dưới.
- Báo repository chưa có code mới: kiểm tra `GIT_BRANCH='HuyenHoang'` và chạy lại cell clone.
- Colab bị ngắt: checkpoint `last.pt` vẫn nằm trên Drive, nhưng script hiện tại chưa resume optimizer; chạy lại sẽ train mới và ghi đè output cùng seed.